# 06b · N-HiTS, TCN, DeepAR y Chronos (Cisco)

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "data.py").exists():
    if ROOT == ROOT.parent:
        raise RuntimeError("Ejecutar desde el repositorio DL-LAB09")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import json
import pandas as pd
from IPython.display import display, Image, Markdown
from src import data as D, metrics as M, backtest as B
RUN_TRAIN = False  # Cambiar a True solo para generar artefactos que falten.


NeuralForecast se ajusta con early stopping temporal y se reentrena con train+validación. Chronos-Bolt-small es zero-shot en CPU. Todos los modelos pasan por `B.backtest` y conservan los mismos clientes/orígenes.

In [2]:
if RUN_TRAIN:
    import subprocess
    subprocess.run([sys.executable,"-m","src.run_cisco","--stage","libraries"],cwd=ROOT,check=True)

In [3]:
if RUN_TRAIN:
    import subprocess
    subprocess.run([sys.executable,"-m","src.run_cisco","--stage","chronos"],cwd=ROOT,check=True)

In [4]:
names=['NHITS', 'TCN', 'DeepAR', 'ChronosBolt']
rows=[]
for name in names:
    path=ROOT / "outputs" / "results" / f"{name}.json"
    if not path.exists():
        raise FileNotFoundError(f"Falta {path}: ejecutar la etapa correspondiente")
    rows.append(json.loads(path.read_text()))
display(pd.DataFrame(rows).set_index("modelo"))

,hardware,inference_s,familia,L,estrategia,covariables,params_total,params_entrenables,train_s,pasos_seleccion,...,validation_step_size,pinball_q10,pinball_q50,pinball_q90,pinball,cobertura_P10_P90,model_id,load_s,cuantiles_postproceso,contaminacion_preentrenamiento
modelo,,,,,,,,,,,,,,,,,,,,,
NHITS,CPU (AMD Ryzen AI 7 350 w/ Radeon 860M),30.961770,Convolucionales,168,multi-salida,False,94626,94626,83.497786,250.0,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
TCN,CPU (AMD Ryzen AI 7 350 w/ Radeon 860M),33.539814,Convolucionales,168,multi-salida,True,21689,21689,209.264232,250.0,...,24.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
DeepAR,CPU (AMD Ryzen AI 7 350 w/ Radeon 860M),31.721966,Probabilísticos,168,probabilística AR,True,5445,5442,154.044209,250.0,...,24.0,36.769054,92.275050,47.472543,58.838882,0.711941,NaN,NaN,NaN,NaN
ChronosBolt,CPU (AMD Ryzen AI 7 350 w/ Radeon 860M),84.806595,Fundacional,168,zero-shot,False,47718016,0,0.000000,NaN,...,NaN,14.289627,28.597483,14.023352,18.970154,0.812014,amazon/chronos-bolt-small,2.890486,reordenamiento monótono de q10/q50/q90 por cli...,Electricity puede formar parte del preentrenam...


In [5]:
display(pd.read_csv(ROOT/"outputs"/"tables"/"probabilistica.csv"))

,modelo,pinball_q10,pinball_q50,pinball_q90,pinball,cobertura_P10_P90
0,ChronosBolt,14.289627,28.597483,14.023352,18.970154,0.812014
1,DeepAR,36.769054,92.275050,47.472543,58.838882,0.711941


DeepAR obtiene P10/P50/P90 de trayectorias; Chronos los estima directamente y aplica reordenamiento monótono por hora. La cobertura nominal P10–P90 es 80%. No se verifica exclusión de Electricity del preentrenamiento de Chronos. Fuentes y detalles: `informe/investigacion.md`.